In [2]:
import pandas as pd

matches=pd.read_csv('data/matches.csv')
deliveries=pd.read_csv('data/deliveries.csv')

print(matches.shape,deliveries.shape)
print(matches.columns.tolist())
print(deliveries.columns.tolist())

matches.info()
matches.isnull().sum()

(39, 23) (17477, 19)
['match_id', 'date', 'venue', 'team1', 'team2', 'stage', 'toss_winner', 'toss_decision', 'first_ings_score', 'first_ings_wkts', 'second_ings_score', 'second_ings_wkts', 'match_result', 'match_winner', 'wb_runs', 'wb_wickets', 'balls_left', 'player_of_the_match', 'top_scorer', 'highscore', 'best_bowling', 'best_bowling_figure', 'super_over_match']
['match_no', 'date', 'stage', 'venue', 'batting_team', 'bowling_team', 'innings', 'over', 'striker', 'bowler', 'runs_of_bat', 'extras', 'wide', 'legbyes', 'byes', 'noballs', 'wicket_type', 'player_dismissed', 'fielder']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   match_id             39 non-null     int64  
 1   date                 39 non-null     object 
 2   venue                39 non-null     object 
 3   team1                39 non-null     object 
 4   te

match_id                0
date                    0
venue                   0
team1                   0
team2                   0
stage                   0
toss_winner             0
toss_decision           0
first_ings_score        1
first_ings_wkts         1
second_ings_score       1
second_ings_wkts        1
match_result            0
match_winner            1
wb_runs                24
wb_wickets             17
balls_left              1
player_of_the_match     1
top_scorer              1
highscore               1
best_bowling            1
best_bowling_figure     1
super_over_match        0
dtype: int64

In [3]:
matches[matches['match_result'].isnull() | matches['first_ings_score'].isnull()]

,match_id,date,venue,team1,team2,stage,toss_winner,toss_decision,first_ings_score,first_ings_wkts,...,match_winner,wb_runs,wb_wickets,balls_left,player_of_the_match,top_scorer,highscore,best_bowling,best_bowling_figure,super_over_match
11,12,"April 06, 2026","Eden Gardens, Kolkata",KKR,PBKS,League,KKR,Bat,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No


In [4]:
import sqlite3

conn = sqlite3.connect('ipl.db')
matches.to_sql('matches', conn, if_exists='replace', index=False)
deliveries.to_sql('deliveries', conn, if_exists='replace', index=False)

17477

In [5]:
query1 = """
SELECT striker, SUM(runs_of_bat) AS total_runs
FROM deliveries
GROUP BY striker
ORDER BY total_runs DESC
LIMIT 10
"""
pd.read_sql(query1, conn)

,striker,total_runs
0,Vaibhav Sooryavanshi,776
1,Shubman Gill,732
2,Sai Sudharsan,722
3,Virat Kohli,675
4,Heinrich Klaasen,624
5,Ishan Kishan,604
6,Rahul,593
7,Mitchell Marsh,563
8,Abhishek Sharma,563
9,Jos Buttler,526


In [17]:
query2 = """
SELECT striker,
       SUM(runs_of_bat) AS total_runs,
       COUNT(*) AS balls_faced,
       ROUND(SUM(runs_of_bat) * 100.0 / COUNT(*), 2) AS strike_rate
FROM deliveries
WHERE wide = 0
GROUP BY striker
HAVING balls_faced >= 30
ORDER BY strike_rate DESC
LIMIT 10
"""
pd.read_sql(query2, conn)

,striker,total_runs,balls_faced,strike_rate
0,Vaibhav Sooryavanshi,776,327,237.31
1,Finn Allen,349,163,214.11
2,Priyansh Arya,364,172,211.63
3,Abhishek Sharma,563,275,204.73
4,Urvil Patel,129,64,201.56
5,Rajat Patidar,500,259,193.05
6,Shashank Singh,132,70,188.57
7,Tim David,305,162,188.27
8,Ryan Rickelton,448,240,186.67
9,Venkatesh Iyer,209,112,186.61


In [7]:
query3 = """
SELECT team,
       COUNT(*) AS matches_played,
       SUM(CASE WHEN team = match_winner THEN 1 ELSE 0 END) AS wins,
       ROUND(SUM(CASE WHEN team = match_winner THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS win_pct
FROM (
    SELECT match_id, team1 AS team, match_winner FROM matches
    UNION ALL
    SELECT match_id, team2 AS team, match_winner FROM matches
)
GROUP BY team
ORDER BY win_pct DESC
"""
pd.read_sql(query3, conn)

,team,matches_played,wins,win_pct
0,PBKS,7,6,85.71
1,RCB,8,6,75.00
2,SRH,8,5,62.50
3,RR,8,4,50.00
4,GT,8,4,50.00
5,LSG,8,3,37.50
6,DC,8,3,37.50
7,CSK,8,3,37.50
8,MI,7,2,28.57
9,KKR,8,2,25.00


In [8]:
query4 = """
SELECT venue,
       COUNT(*) AS matches,
       ROUND(AVG(first_ings_score), 1) AS avg_first_innings_score
FROM matches
WHERE first_ings_score IS NOT NULL
GROUP BY venue
ORDER BY avg_first_innings_score DESC
"""
pd.read_sql(query4, conn)


,venue,matches,avg_first_innings_score
0,"Sawai Mansingh Stadium, Jaipur",1,228.0
1,"Wankhede Stadium, Mumbai",4,215.5
2,"New PCA Cricket Stadium, Mullanpur",3,211.7
3,"MA Chidambaram Stadium, Chennai",3,204.3
4,"Rajiv Gandhi International Stadium, Hyderabad",4,202.0
5,"M. Chinnaswamy Stadium, Bangalore",5,195.4
6,"Eden Gardens, Kolkata",3,187.3
7,"Narendra Modi Stadium, Ahmedabad",4,186.8
8,"Arun Jaitley Stadium, Delhi",4,177.8
9,"Barsapara Stadium, Guwahati",3,159.3


In [9]:
query5 = """
SELECT bowler,
       COUNT(*) AS wickets
FROM deliveries
WHERE wicket_type IS NOT NULL AND wicket_type NOT IN ('run out', 'retired hurt')
GROUP BY bowler
ORDER BY wickets DESC
LIMIT 10
"""
pd.read_sql(query5, conn)

,bowler,wickets
0,Kagiso Rabada,29
1,Jofra Archer,29
2,Bhuvneshwar Kumar,29
3,Anshul Kamboj,24
4,Rashid Khan,21
5,Eshan Malinga,21
6,Prince Yadav,20
7,Mohammed Siraj,20
8,Rasikh Salam Dar,19
9,Kartik Tyagi,19


In [18]:
pd.read_sql(query1, conn).to_csv('top_scorers.csv', index=False)
pd.read_sql(query2, conn).to_csv('best_strike_rates.csv', index=False)
pd.read_sql(query3, conn).to_csv('team_win_pct.csv', index=False)
pd.read_sql(query4, conn).to_csv('venue_avg_scores.csv', index=False)
pd.read_sql(query5, conn).to_csv('top_wicket_takers.csv', index=False)